Design and train a RNN using Tensorflow and Keras


In [5]:
#step-1
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"amritpratya","key":"4580f955dfce926805f83384856b0a8d"}'}

In [7]:
#step-2
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [8]:
#step-3
!kaggle datasets download -d soumendraprasad/musical-instruments-sound-dataset
!unzip -q musical-instruments-sound-dataset.zip -d /content/musical-instrument_data

Dataset URL: https://www.kaggle.com/datasets/soumendraprasad/musical-instruments-sound-dataset
License(s): CC0-1.0
 99% 5.34G/5.40G [01:06<00:03, 18.6MB/s]
100% 5.40G/5.40G [01:06<00:00, 87.5MB/s]


In [9]:
#step-4
!pip install librosa soundfile tensorflow pandas

In [10]:
#step-5
import os
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from google.colab import files #for interactive file upload
from google.colab import drive #for mounting Google Drive

In [11]:
#step-6
CSV_PATH = '/content/musical-instrument_data/Metadata_Train.csv'
DATA_PATH = '/content/musical-instrument_data/Train_submission/Train_submission'

train_meta = pd.read_csv(CSV_PATH)
LABELS = ['guitar','drums','violin','piano']

X, y = [], []

for i, row in train_meta.iterrows():
    file_path = os.path.join(DATA_PATH, row['FileName'])
    label = row['Class'].strip().lower()
    try :
        signal, sr = librosa.load(file_path, duration=3.0)
        mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=40).T
        if mfcc.shape[0] >= 130:
            mfcc = mfcc[:130]
            X.append(mfcc)
            y.append(label)
    except Exception as e:
       print(f"Error processing file {file_path}: {e}")

X = np.array(X)
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(X, y_encoded, test_size=0.2, stratify=y_encoded)


In [12]:
#step-7
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(128, input_shape=(130, 40)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(len(le.classes_), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=15, batch_size=32)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - accuracy: 0.4148 - loss: 1.2303 - val_accuracy: 0.6197 - val_loss: 0.7771
Epoch 2/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6083 - loss: 0.7727 - val_accuracy: 0.6325 - val_loss: 0.6258
Epoch 3/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6391 - loss: 0.6258 - val_accuracy: 0.6474 - val_loss: 0.5575
Epoch 4/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6593 - loss: 0.5567 - val_accuracy: 0.6688 - val_loss: 0.5569
Epoch 5/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6558 - loss: 0.5479 - val_accuracy: 0.6047 - val_loss: 0.5269
Epoch 6/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.6953 - loss: 0.5126 - val_accuracy: 0.6282 - val_loss: 0.5129
Epoch 7/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7140 - loss: 0.4824 - val_accuracy: 0.6197 - val_loss: 0.5216
Epoch 8/15
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6835 - loss: 0.5052 - val_accuracy: 0.6517 - v

In [17]:
#step-8
uploaded = files.upload()
for file_name in uploaded.keys():
  try:
    signal, sr = librosa.load(file_name, duration=3.0)
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=40).T

    if mfcc.shape[0] < 130:
      mfcc = np.pad(mfcc,((0, 130 - mfcc.shape[0], (0, 0))))
    else:
      mfcc = mfcc[:130]

    mfcc = mfcc.reshape(1, 130, 40)
    prediction = model.predict(mfcc)
    confidence = np.max(prediction)

    if confidence < 0.75:
      print("Prediction: UNKNOWN or INVALID instrument")
    else:
      predicted_label = le.inverse_transform([np.argmax(prediction)])
      print(f"Prediction: {predicted_label[0]} ({confidence * 100:.2f}% confidence)")
  except Exception as e:
    print(f"Error processing file {file_name}: {e}")


Saving guitar-intro-110935.wav to guitar-intro-110935 (3).wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Prediction: sound_piano (98.63% confidence)
